# RAPID response prediction: expanded grouped-CV grid

This version locates the repository automatically, so it also works when Jupyter starts in `RAPID/notebooks` or another subdirectory.

## Candidate predictors

Resistance / footprint: `len_fraction_width_weighted_log_lambda`, `length_fraction`, `width_fraction`, `k_ratio_to_matched_base`, `along_network_frac`.

Geometry / topology: `equivalent_length`, `elongation`, `topologic_complexity_score`, `effective_n_paths_width`, `dominant_width_fraction`, `path_length_cv`, `branching_density_by_length`.

Independent-collapse topology values come from the matching base-state unit. Compound-bubble values are equivalent-width-weighted means of member-unit values.

The search verifies before fitting that `width_fraction` is the shared domain-aligned exposure factor and that `len_fraction_width_weighted_log_lambda` equals `length_fraction × -log10(Lambda_matched) × width_fraction`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'RAPID/classification/search_rapid_response_predictors.py').is_file():
            return candidate
    raise FileNotFoundError('Could not find river-hierarchy repo. Set REPO_ROOT manually.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11', '13','15', '21','22','37','41','45','46','48','54', '55', '56']
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = REPO_ROOT / 'RAPID/classification/results' / f'rapid-response-grid-v1-{STAMP}'

assert OUTPUT_ROOT.exists(), f'Missing RAPID output root: {OUTPUT_ROOT}'
print('Repository root:', REPO_ROOT)
RUN_DIR

## Run the search

This evaluates 794 predictor sets (all one- through four-variable combinations, plus the full 12-variable set), 12 model settings, eight leave-one-example-out folds, and two separate response targets. It can take a substantial time.

In [ ]:
command = [
    sys.executable, str(REPO_ROOT / 'RAPID/classification/search_rapid_response_predictors.py'),
    '--output-root', str(OUTPUT_ROOT), '--results-dir', str(RUN_DIR),
    '--example-ids', *EXAMPLE_IDS,
]
completed = subprocess.run(command, cwd=REPO_ROOT, check=True, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

## Compare screening candidates

Lower MAE is better. These are screening rankings, rather than final unbiased scores: use physically interpretable, similarly scoring candidates as the preferred model, and validate on new examples.

In [ ]:
scores = pd.read_csv(RUN_DIR / 'grouped_cv_grid_scores.csv').sort_values(['r2','target', 'mae_pct_points', 'n_features', 'rmse_pct_points'], ascending=[False, True, True, True, True])
for target, table in scores.groupby('target', sort=False):
    print(f'\n{target}')
    display(table[['model', 'features','mae_pct_points','r2', 'n_features']].head(15).reset_index(drop=True))

In [ ]:
cohort = pd.read_csv(RUN_DIR / 'common_complete_case_cohort.csv')
provenance = pd.read_csv(RUN_DIR / 'topology_metric_provenance.csv')
exposure_audit = pd.read_csv(RUN_DIR / 'exposure_metric_audit.csv')
print(f'Complete-case states: {len(cohort)} across {cohort.example_id.nunique()} examples')
display(cohort.groupby(['example_id', 'mode']).size().rename('states').reset_index())
display(provenance)
print('Domain-aligned exposure audit (verified before model fitting):')
display(exposure_audit)

## Observed versus held-out predictions

Every point is predicted by a model that did not train on that example.

In [ ]:
predictions = pd.read_csv(RUN_DIR / 'grouped_cv_grid_predictions.csv')
best = scores.groupby('target', as_index=False).first()
fig, axes = plt.subplots(1, len(best), figsize=(7 * len(best), 5), squeeze=False)
for axis, row in zip(axes.ravel(), best.itertuples(index=False)):
    plot_data = predictions[(predictions.target == row.target) & (predictions.model == row.model) & (predictions.features == row.features)]
    axis.scatter(plot_data.observed, plot_data.predicted, alpha=0.75)
    limits = [min(plot_data.observed.min(), plot_data.predicted.min()), max(plot_data.observed.max(), plot_data.predicted.max())]
    axis.plot(limits, limits, '--', color='black', linewidth=1)
    axis.set(xlabel='Observed response (% change)', ylabel='Held-out prediction (% change)', title=f'{row.target}\nMAE {row.mae_pct_points:.3f}; {row.model}')
plt.tight_layout()

## Negative-shift mechanism: diagnostic figures and planned conditional tests

### What remains primary

The grid above is the **primary all-state analysis**. Its target is the raw signed timing shift, `lag_to_inflow_peak_pct_shift_vs_base`; it retains negative, zero, and positive shifts. Its MAE and R² are evaluated in original percentage points.

### What this section diagnoses

The second figure excludes zero and positive shifts and asks a narrower physical question: *when a collapse advances the outlet peak, how large is that advance?* For a negative signed shift `ΔL`, the plotted magnitude is `|ΔL| = -ΔL`. The logarithmic y-axis reveals structure among small non-zero advances, but it is a diagnostic figure only—not a signed-response plot and not a log-space R².

In [ ]:
TIMING_TARGET = 'lag_to_inflow_peak_pct_shift_vs_base'
PRIMARY_PREDICTOR = 'len_fraction_width_weighted_log_lambda'

signed = cohort[[PRIMARY_PREDICTOR, TIMING_TARGET, 'mode', 'example_id']].dropna().copy()
negative = signed.loc[signed[TIMING_TARGET] < 0].copy()
negative['advance_magnitude_pct'] = -negative[TIMING_TARGET]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for mode, part in signed.groupby('mode'):
    axes[0].scatter(part[PRIMARY_PREDICTOR], part[TIMING_TARGET], alpha=0.7, label=mode)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set(
    xlabel='Length × −log10(Λ matched) × domain-aligned exposure',
    ylabel='Signed lag shift versus base (percentage points)',
    title='Primary context: all signed timing shifts',
)
axes[0].legend(title='Collapse mode')

for mode, part in negative.groupby('mode'):
    axes[1].scatter(part[PRIMARY_PREDICTOR], part['advance_magnitude_pct'], alpha=0.7, label=mode)
axes[1].set_yscale('log')
axes[1].set(
    xlabel='Length × −log10(Λ matched) × domain-aligned exposure',
    ylabel='Advance magnitude, −ΔL (percentage points; log scale)',
    title='Magnitude-only diagnostic: negative shifts only',
)
axes[1].legend(title='Collapse mode')
plt.tight_layout()
print(f'All signed states shown: {len(signed)}; negative-shift states shown in log diagnostic: {len(negative)}.')

### Planned conditional-model run — not run by this notebook yet

If the diagnostic remains convincing, conduct this as a separate experiment rather than replacing the primary grid:

1. **All-state feature-transform control:** retain the signed target and compare selected physically justified predictor transforms with the current raw-predictor model.
2. **Conditional magnitude model:** retain only negative lag shifts and model `advance_magnitude_pct = -ΔL` on its original scale.
3. **Conditional log-magnitude model:** use `log1p(advance_magnitude_pct / s)` for a named physical scale `s`, back-transform predictions, then report MAE/R² in original percentage points.
4. **Sensitivity:** repeat conditional runs for `ΔL < 0`, `< -0.05`, and `< -0.10` percentage points.

All of these must preserve leave-one-example-out validation. A later two-stage model would first predict advance versus no advance, then predict magnitude only for predicted advances; this is necessary before a conditional model can be used prospectively.

In [ ]:
# This is a recorded design for a future, separate run. It starts no model fitting.
FUTURE_CONDITIONAL_RUN = {
    'primary_analysis': 'all states; signed percentage-point lag shift',
    'conditional_population': 'lag_to_inflow_peak_pct_shift_vs_base < threshold',
    'thresholds_pct_points': [0.0, -0.05, -0.10],
    'conditional_target': 'advance_magnitude_pct = -lag_shift_pct',
    'response_variants': ['raw advance magnitude', 'log1p(advance magnitude / s), with inverse transform'],
    'candidate_predictor_transforms': ['log(equivalent_length)', 'log1p(k_ratio_to_matched_base)', 'log(elongation)', 'log(effective_n_paths_width)', 'log(topologic_complexity_score)', 'log1p(branching_density_by_length)'],
    'validation': 'leave one example_id out; score predictions back on original percentage-point scale',
}
pd.Series(FUTURE_CONDITIONAL_RUN, name='planned test').to_frame()

## Quick preliminary result: one fixed Ridge model

Use this section after a kernel restart when you need results for a selected example set without running the broad grid. It runs **only** the model selected for its strong grouped-CV R²:

`Ridge(alpha=10)` with `len_fraction_width_weighted_log_lambda`, `equivalent_length`, and `topologic_complexity_score`.

This is a preliminary, fixed-model evaluation—not fresh model selection. It retains the shared domain-aligned exposure verification and uses leave-one-example-out validation.

In [ ]:
# Change this list to the RAPID examples you want in the preliminary result.
PRELIM_EXAMPLE_IDS = ['02', '03', '05', '11', '13','15', '21','22','37','41','45','46','48','54', '55', '56']
PRELIM_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PRELIM_RUN_DIR = REPO_ROOT / 'RAPID/classification/results' / f'rapid-response-preliminary-ridge-v1-{PRELIM_STAMP}'

preliminary_command = [
    sys.executable, str(REPO_ROOT / 'RAPID/classification/run_rapid_response_preliminary_model.py'),
    '--output-root', str(OUTPUT_ROOT), '--results-dir', str(PRELIM_RUN_DIR),
    '--example-ids', *PRELIM_EXAMPLE_IDS,
]
try:
    completed = subprocess.run(preliminary_command, cwd=REPO_ROOT, check=True, text=True, capture_output=True)
except subprocess.CalledProcessError as error:
    print(error.stdout or '')
    print(error.stderr or '')
    raise
else:
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)

In [ ]:
preliminary_score = pd.read_csv(PRELIM_RUN_DIR / 'grouped_cv_score.csv')
preliminary_predictions = pd.read_csv(PRELIM_RUN_DIR / 'grouped_cv_predictions.csv')
display(preliminary_score)

preliminary_predictions['observed_lag_shift_pct'] = preliminary_predictions['observed_lag_shift_pct'] *-1
preliminary_predictions['predicted_lag_shift_pct'] = preliminary_predictions['predicted_lag_shift_pct'] *-1

fig, axis = plt.subplots(figsize=(6, 6))
axis.scatter(preliminary_predictions['observed_lag_shift_pct'], preliminary_predictions['predicted_lag_shift_pct'], alpha=0.75)
limits = [
    min(preliminary_predictions['observed_lag_shift_pct'].min(), preliminary_predictions['predicted_lag_shift_pct'].min()),
    max(preliminary_predictions['observed_lag_shift_pct'].max(), preliminary_predictions['predicted_lag_shift_pct'].max()),
]
axis.plot(limits, limits, '--', color='black', linewidth=1)
axis.set(xlabel='RAPID predicted lag shift (percentage points)', ylabel='Geometry predicted lag shift (percentage points)', 
         title='Fixed Ridge model')
axis.set_yscale('log')
axis.set_xscale('log')
plt.tight_layout()